# MSDS 565 — Unit 1, Notebook 2: Preprocessing
**Team DataStars**
This notebook builds a clean, model-ready version of the CarSales Large dataset and exports the exact 80/20 split reused in Notebook 3.

### Changes incorporated
- Removed stale/duplicate scratch cells and corrected markdown counts so outputs, not hand-typed totals, are the source of truth.
- Preserved the validated Section 2 decisions on `horsepower`, `power`/`torque`, and `engine_type`/`engine_cylinders`.
- Parsed `bed_length` as a numeric measurement instead of accidentally treating it as a categorical feature.
- Replaced the blanket `$200,000` price cutoff with the contextual price-quality review.
- Retained plausible classic years and plausible extreme luxury/exotic prices.
- Kept the `is_new` field unchanged when it conflicts with mileage; a separate conflict indicator is engineered instead of overwriting the source value.
- Moved the 80/20 split **before any learned imputation, category-frequency selection, or scaling**. All learned preprocessing statistics are fit on the training split and then applied to the test split to avoid train/test contamination.
- Excluded `price` from every imputation/feature-engineering step and from scaling.
- Preserved original object-dtype metadata for Notebook 5 fairness work.


## 0. Setup
The raw CSV is read fresh in this notebook; no objects are carried over from Notebook 1. `price` is the regression target and is treated as off-limits for feature engineering.


In [2]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# Notebook is expected to live in the repo's notebooks/ folder.
# Update only this path if your local folder structure differs.
DATA_PATH = Path(r"C:\Users\india\OneDrive\Desktop\MSDS 565\MSDS565-CarSales-DataStars\data\CarSales_Large.csv")
OUTPUT_DIR = DATA_PATH.parent  # local class data folder — not committed to the repo


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Update DATA_PATH to your local CarSales_Large.csv."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
n_raw = len(df)
print(f"Raw shape: {df.shape[0]:,} rows x {df.shape[1]} columns")


Raw shape: 302,590 rows x 66 columns


## 1. Dropping
### 1.1 Columns
The following columns are removed because they are identifiers, unusable free text/images for this tabular assignment, entirely missing, redundant with a cleaner field, or direct price leakage. `trimId` is a high-cardinality coded identifier; its useful vehicle information is represented more interpretably by `trim_name` and `model_name`.


In [3]:
drop_cols = [
    "Unnamed: 0",             # row-index artifact
    "listing_id", "sp_id",    # identifiers
    "trimId",                 # high-cardinality coded identifier
    "main_picture_url",       # image URL; image modeling is out of scope
    "description",            # free text; NLP is out of scope here
    "combine_fuel_economy", "is_certified", "vehicle_damage_category",  # 100% missing
    "bed_height",             # placeholder-only / unusable
    "wheel_system_display",   # redundant with wheel_system
    "dealer_zip",             # redundant with available geographic coordinates
    "savings_amount",         # directly tied to price -> target leakage risk
]

drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

print(f"Dropped {len(drop_cols)} columns: {drop_cols}")
print(f"Shape after column drops: {df.shape}")


Dropped 13 columns: ['Unnamed: 0', 'listing_id', 'sp_id', 'trimId', 'main_picture_url', 'description', 'combine_fuel_economy', 'is_certified', 'vehicle_damage_category', 'bed_height', 'wheel_system_display', 'dealer_zip', 'savings_amount']
Shape after column drops: (302590, 53)


### 1.2 Exact duplicate modeling records
Notebook 1 found 1,210 duplicates when only two key columns were ignored. After all non-modeling columns above are removed, additional rows become identical across the actual modeling fields. We therefore recompute duplicates **after** the full column drop and remove the exact duplicates that remain. This prevents identical modeling records from receiving extra weight or appearing on both sides of the later train/test split.


In [4]:
n_before_dupes = len(df)
df = df.drop_duplicates().copy()
n_dupes_dropped = n_before_dupes - len(df)

print(f"Dropped {n_dupes_dropped:,} exact duplicate modeling records")
print(f"Rows remaining: {len(df):,}")
print(f"Percent of raw rows removed so far: {(n_raw - len(df)) / n_raw * 100:.2f}%")


Dropped 5,994 exact duplicate modeling records
Rows remaining: 296,596
Percent of raw rows removed so far: 1.98%


## 2. Data Types and Redundancy Validation
### 2.1 Convert `"--"` placeholders to real missing values
The literal string `"--"` is used as a missing-value placeholder in several text columns. It is converted to `NaN` so downstream missingness checks and imputation see the true missing-data pattern.


In [5]:
obj_cols_initial = df.select_dtypes("object").columns
placeholder_hits = 0

for c in obj_cols_initial:
    mask = df[c] == "--"
    if mask.any():
        placeholder_hits += int(mask.sum())
        df.loc[mask, c] = np.nan

print(
    f"Replaced {placeholder_hits:,} '--' placeholders with NaN "
    f"across {len(obj_cols_initial)} object columns"
)


Replaced 12,000 '--' placeholders with NaN across 39 object columns


### 2.2 Normalize Boolean flags
`is_cpo` and `is_oemcpo` are one-sided flags: presence indicates `True` and missing indicates `False`. The history-report fields are different: missing means the report information was unavailable, not that the event definitely did not occur. Those fields remain nullable until Section 4, where missingness is explicitly encoded.


In [6]:
one_sided_bool_cols = [c for c in ["is_cpo", "is_oemcpo"] if c in df.columns]
for c in one_sided_bool_cols:
    df[c] = df[c].notna()

clean_bool_cols = [c for c in ["is_new", "franchise_dealer"] if c in df.columns]
for c in clean_bool_cols:
    df[c] = df[c].astype("boolean")

history_block_cols = [
    c for c in ["fleet", "frame_damaged", "has_accidents", "isCab", "salvage", "theft_title"]
    if c in df.columns
]
for c in history_block_cols:
    df[c] = df[c].astype("boolean")

print("One-sided flags (NaN -> False):", one_sided_bool_cols)
print("Nullable clean booleans:", clean_bool_cols)
print("History-report booleans (missing retained):", history_block_cols)


One-sided flags (NaN -> False): ['is_cpo', 'is_oemcpo']
Nullable clean booleans: ['is_new', 'franchise_dealer']
History-report booleans (missing retained): ['fleet', 'frame_damaged', 'has_accidents', 'isCab', 'salvage', 'theft_title']


### 2.3 Parse numeric measurements and temporarily parse `power` / `torque`
Unit-bearing measurements are converted to numeric values. `bed_length` is included here because it is a physical measurement, not a categorical feature. `power` and `torque` are temporarily parsed into value/RPM components so their reliability can be validated before a keep/drop decision is made.


In [7]:
def extract_number(series):
    return pd.to_numeric(
        series.astype(str).str.extract(r"(-?\d+\.?\d*)")[0],
        errors="coerce"
    )

unit_cols = [
    "back_legroom", "front_legroom", "fuel_tank_volume",
    "height", "length", "wheelbase", "width", "maximum_seating",
    "bed_length"
]
unit_cols = [c for c in unit_cols if c in df.columns]

for c in unit_cols:
    df[c] = extract_number(df[c])

# Temporarily keep original power/torque strings for validation.
for c, val_name, rpm_name in [
    ("power", "power_hp", "power_rpm"),
    ("torque", "torque_lbft", "torque_rpm")
]:
    if c in df.columns:
        parts = df[c].astype(str).str.extract(
            r"(?P<val>-?\d[\d,]*\.?\d*)\D+(?P<rpm>[\d,]+)"
        )
        df[val_name] = pd.to_numeric(
            parts["val"].str.replace(",", "", regex=False), errors="coerce"
        )
        df[rpm_name] = pd.to_numeric(
            parts["rpm"].str.replace(",", "", regex=False), errors="coerce"
        )

print("Parsed measurement columns:", unit_cols)
print(
    "Temporary power/torque numeric fields:",
    [c for c in ["power_hp", "power_rpm", "torque_lbft", "torque_rpm"] if c in df.columns]
)


Parsed measurement columns: ['back_legroom', 'front_legroom', 'fuel_tank_volume', 'height', 'length', 'wheelbase', 'width', 'maximum_seating', 'bed_length']
Temporary power/torque numeric fields: ['power_hp', 'power_rpm', 'torque_lbft', 'torque_rpm']


#### 2.3.1 Torque parser spot-check
This verifies the regex itself. A successful parse does **not** prove that the source specification belongs to the correct engine; that is checked separately below.


In [8]:
if {"torque", "torque_lbft", "torque_rpm"}.issubset(df.columns):
    torque_nonmissing = df[["torque", "torque_lbft", "torque_rpm"]].dropna()
    n_show = min(8, len(torque_nonmissing))
    display(torque_nonmissing.sample(n=n_show, random_state=42))


,torque,torque_lbft,torque_rpm
101304,"369 lb-ft @ 1,370 RPM",369.0,1370.0
122671,"175 lb-ft @ 4,400 RPM",175.0,4400.0
226984,"172 lb-ft @ 4,100 RPM",172.0,4100.0
93677,"186 lb-ft @ 4,000 RPM",186.0,4000.0
283253,"263 lb-ft @ 4,700 RPM",263.0,4700.0
215294,"174 lb-ft @ 4,000 RPM",174.0,4000.0
54254,"107 lb-ft @ 4,400 RPM",107.0,4400.0
189571,"148 lb-ft @ 1,850 RPM",148.0,1850.0


#### 2.3.2 Horsepower consistency check
The dataset contains both a numeric `horsepower` feature and horsepower embedded in the `power` string. Because the two represent the same underlying characteristic, they are compared before deciding which source to retain.


In [9]:
hp_compare = df.loc[
    df["horsepower"].notna() & df["power_hp"].notna(),
    ["horsepower", "power_hp"]
].copy()

hp_compare["difference"] = (hp_compare["horsepower"] - hp_compare["power_hp"]).abs()

print(f"Rows with both values: {len(hp_compare):,}")
print(f"Exact matches: {(hp_compare['difference'] == 0).sum():,}")
print(f"Different values: {(hp_compare['difference'] != 0).sum():,}")
print(f"Percent matching: {(hp_compare['difference'] == 0).mean() * 100:.2f}%")

display(hp_compare["difference"].describe())
print()
print("Difference ranges:")
print("1-5 HP:", hp_compare["difference"].between(1, 5).sum())
print("6-10 HP:", hp_compare["difference"].between(6, 10).sum())
print("11-25 HP:", hp_compare["difference"].between(11, 25).sum())
print("26-50 HP:", hp_compare["difference"].between(26, 50).sum())
print("More than 50 HP:", (hp_compare["difference"] > 50).sum())


Rows with both values: 250,830
Exact matches: 226,482
Different values: 24,348
Percent matching: 90.29%


count    250830.000000
mean          5.760081
std          21.943291
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         180.000000
Name: difference, dtype: float64


Difference ranges:
1-5 HP: 1418
6-10 HP: 1898
11-25 HP: 4818
26-50 HP: 3329
More than 50 HP: 12885


In [10]:
hp_mismatch = df.loc[
    df["horsepower"].notna()
    & df["power_hp"].notna()
    & (df["horsepower"] != df["power_hp"])
].copy()

hp_mismatch["hp_difference"] = (
    hp_mismatch["horsepower"] - hp_mismatch["power_hp"]
).abs()

hp_cols = [
    "make_name", "model_name", "trim_name", "year",
    "engine_type", "engine_displacement",
    "horsepower", "power_hp", "power_rpm", "hp_difference"
]

display(
    hp_mismatch[hp_cols]
    .sort_values("hp_difference", ascending=False)
    .head(8)
)


,make_name,model_name,trim_name,year,engine_type,engine_displacement,horsepower,power_hp,power_rpm,hp_difference
272330,Dodge,RAM 1500,ST LB RWD,2009,V6,3700.0,210.0,390.0,5600.0,180.0
294935,Dodge,RAM 1500,ST Quad Cab RWD,2009,V6,3700.0,210.0,390.0,5600.0,180.0
209518,Dodge,RAM 1500,ST Quad Cab RWD,2010,V6,3700.0,215.0,390.0,5600.0,175.0
235136,Dodge,RAM 1500,ST Quad Cab RWD,2010,V6,3700.0,215.0,390.0,5600.0,175.0
63497,RAM,1500,ST Quad Cab,2012,V6,3700.0,215.0,390.0,5600.0,175.0
123092,RAM,1500,ST Quad Cab,2011,V6,3700.0,215.0,390.0,5600.0,175.0
299573,RAM,1500,ST,2011,V6,3700.0,215.0,390.0,5600.0,175.0
189546,RAM,1500,ST Quad Cab,2011,V6,3700.0,215.0,390.0,5600.0,175.0


**Decision:** The two horsepower sources match for most records, but the disagreements include large, systematic differences. The original `horsepower` column has greater coverage and aligns more consistently with the listed engine details in the inspected mismatches. `horsepower` is retained; parsed `power_hp` is not retained as a separate predictor. Missing `horsepower` is handled later using training-only, vehicle-aware imputation.


#### 2.3.3 Torque/source consistency check
The parser extracts torque strings correctly, but the torque values on the largest horsepower-mismatch vehicles show the same source-alignment concern as `power`. The problem is therefore treated as a source-data reliability issue rather than a regex issue.


In [11]:
torque_cols = [
    "make_name", "model_name", "trim_name", "year",
    "engine_type", "engine_displacement",
    "horsepower", "power_hp",
    "torque", "torque_lbft", "torque_rpm", "hp_difference"
]

display(
    hp_mismatch[torque_cols]
    .sort_values("hp_difference", ascending=False)
    .head(8)
)


,make_name,model_name,trim_name,year,engine_type,engine_displacement,horsepower,power_hp,torque,torque_lbft,torque_rpm,hp_difference
272330,Dodge,RAM 1500,ST LB RWD,2009,V6,3700.0,210.0,390.0,"407 lb-ft @ 4,000 RPM",407.0,4000.0,180.0
294935,Dodge,RAM 1500,ST Quad Cab RWD,2009,V6,3700.0,210.0,390.0,"407 lb-ft @ 4,000 RPM",407.0,4000.0,180.0
209518,Dodge,RAM 1500,ST Quad Cab RWD,2010,V6,3700.0,215.0,390.0,"407 lb-ft @ 4,000 RPM",407.0,4000.0,175.0
235136,Dodge,RAM 1500,ST Quad Cab RWD,2010,V6,3700.0,215.0,390.0,"407 lb-ft @ 4,000 RPM",407.0,4000.0,175.0
63497,RAM,1500,ST Quad Cab,2012,V6,3700.0,215.0,390.0,"407 lb-ft @ 4,000 RPM",407.0,4000.0,175.0
123092,RAM,1500,ST Quad Cab,2011,V6,3700.0,215.0,390.0,"407 lb-ft @ 4,000 RPM",407.0,4000.0,175.0
299573,RAM,1500,ST,2011,V6,3700.0,215.0,390.0,"407 lb-ft @ 4,000 RPM",407.0,4000.0,175.0
189546,RAM,1500,ST Quad Cab,2011,V6,3700.0,215.0,390.0,"407 lb-ft @ 4,000 RPM",407.0,4000.0,175.0


**Decision:** `power`, `power_hp`, `power_rpm`, `torque`, `torque_lbft`, and `torque_rpm` are not retained for modeling. The parsing worked, but inspection showed that the underlying specifications can be inconsistent with the listed engine configuration. More reliable engine information remains available through `horsepower`, `engine_type`, and `engine_displacement`.


#### 2.3.4 Engine redundancy check
`engine_type` and `engine_cylinders` appeared identical. We confirmed both value agreement and missingness before dropping one.


In [12]:
both_engine = df["engine_type"].notna() & df["engine_cylinders"].notna()
engine_match = (
    df.loc[both_engine, "engine_type"]
    == df.loc[both_engine, "engine_cylinders"]
)

print(f"Rows with both values: {both_engine.sum():,}")
print(f"Exact matches: {engine_match.sum():,}")
print(f"Different values: {(~engine_match).sum():,}")
print(f"Percent matching: {engine_match.mean() * 100:.2f}%")

print()
print("Missingness comparison:")
print("engine_type missing:", df["engine_type"].isna().sum())
print("engine_cylinders missing:", df["engine_cylinders"].isna().sum())
print(
    "engine_type present, engine_cylinders missing:",
    (df["engine_type"].notna() & df["engine_cylinders"].isna()).sum()
)
print(
    "engine_cylinders present, engine_type missing:",
    (df["engine_cylinders"].notna() & df["engine_type"].isna()).sum()
)


Rows with both values: 286,382
Exact matches: 286,382
Different values: 0
Percent matching: 100.00%

Missingness comparison:
engine_type missing: 10214
engine_cylinders missing: 10214
engine_type present, engine_cylinders missing: 0
engine_cylinders present, engine_type missing: 0


#### 2.3.5 Numeric redundancy screen
An absolute correlation of 0.90 is used only as an **investigation threshold**, not an automatic drop rule. Highly correlated variables are retained when they represent distinct concepts (for example, overall length vs. wheelbase, or city vs. highway fuel economy).


In [13]:
numeric_df = df.select_dtypes(include="number").drop(columns=["price"], errors="ignore")
corr_matrix = numeric_df.corr().abs()

high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        correlation = corr_matrix.iloc[i, j]
        if correlation >= 0.90:
            high_corr_pairs.append({
                "feature_1": corr_matrix.columns[i],
                "feature_2": corr_matrix.columns[j],
                "correlation": correlation,
            })

high_corr_df = pd.DataFrame(high_corr_pairs)
if not high_corr_df.empty:
    display(high_corr_df.sort_values("correlation", ascending=False))
else:
    print("No numeric predictor pairs met the 0.90 investigation threshold.")


,feature_1,feature_2,correlation
2,power_hp,horsepower,0.968803
1,wheelbase,length,0.961784
0,highway_fuel_economy,city_fuel_economy,0.941870


**Redundancy conclusion:** `power_hp` is redundant with `horsepower` and has source-quality concerns, so it is removed. `engine_cylinders` is an exact duplicate of `engine_type` in both observed values and missingness, so `engine_type` is retained. `length`/`wheelbase` and `city_fuel_economy`/`highway_fuel_economy` are highly correlated but measure different vehicle characteristics, so both members of those pairs are retained.


In [14]:
cols_to_drop = [
    "power", "power_hp", "power_rpm",
    "torque", "torque_lbft", "torque_rpm",
    "engine_cylinders",
]
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)

print("Dropped redundant/unreliable features:")
print(cols_to_drop)
print("Dataset shape after Section 2:", df.shape)


Dropped redundant/unreliable features:
['power', 'power_hp', 'power_rpm', 'torque', 'torque_lbft', 'torque_rpm', 'engine_cylinders']
Dataset shape after Section 2: (296596, 50)


## 3. Outlier Handling
The assignment requires `mileage`, `price`, and `year` to be handled at minimum. Our review also retains a focused fuel-economy data-quality check. Row loss from null/outlier handling is measured against the assignment's combined 50% ceiling.


In [15]:
rows_start_outliers = len(df)
print(f"Rows at start of Section 3: {rows_start_outliers:,}")


Rows at start of Section 3: 296,596


### 3.1 Mileage
Two observations contain an implausible odometer sentinel near 100 million miles. These values are clearly invalid and strongly inflate mileage dispersion, so the two rows are removed rather than imputed.


In [16]:
mileage_sentinel_mask = df["mileage"] > 500_000
n_mileage_dropped = int(mileage_sentinel_mask.sum())
df = df.loc[~mileage_sentinel_mask].copy()

print(f"Dropped {n_mileage_dropped} mileage-sentinel rows")
print(f"Rows remaining: {len(df):,}")


Dropped 2 mileage-sentinel rows
Rows remaining: 296,594


### 3.2 Price
A fixed upper cutoff is not used. Inspection showed that some globally extreme prices belong to legitimate exotic, luxury, or specialty vehicles. Instead, we screen price contextually against the median for the same make/model/year and remove only a small set of records with strong internal evidence of implausible pricing.


In [17]:
# Global distribution review
print(df["price"].describe(percentiles=[0.90, 0.95, 0.99, 0.995, 0.999]))

print()
print("Listings above selected price thresholds:")
for threshold in [100_000, 150_000, 200_000, 250_000, 500_000, 1_000_000]:
    count = (df["price"] > threshold).sum()
    pct = count / df["price"].notna().sum() * 100
    print(f">${threshold:,.0f}: {count:,} ({pct:.4f}%)")


count    2.965940e+05
mean     2.912954e+04
std      2.059865e+04
min      3.869000e+02
50%      2.508450e+04
90%      5.032500e+04
95%      6.019570e+04
99%      8.782175e+04
99.5%    1.121638e+05
99.9%    2.095183e+05
max      1.750000e+06
Name: price, dtype: float64

Listings above selected price thresholds:
>$100,000: 1,918 (0.6467%)
>$150,000: 699 (0.2357%)
>$200,000: 321 (0.1082%)
>$250,000: 171 (0.0577%)
>$500,000: 15 (0.0051%)
>$1,000,000: 5 (0.0017%)


#### 3.2.1 Contextual price screen
Ratios above 5× or below 20% of the same make/model/year median are used as **screening thresholds only**. This diagnostic helps locate records for review; it is not itself a deletion rule.


In [18]:
price_check = df[[
    "make_name", "model_name", "trim_name", "year", "mileage", "price"
]].copy()

price_check["group_median_price"] = df.groupby(
    ["make_name", "model_name", "year"]
)["price"].transform("median")
price_check["group_size"] = df.groupby(
    ["make_name", "model_name", "year"]
)["price"].transform("size")
price_check["price_to_group_median"] = (
    price_check["price"] / price_check["group_median_price"]
)

price_flags = price_check.loc[
    (price_check["group_size"] >= 5)
    & (
        (price_check["price_to_group_median"] > 5)
        | (price_check["price_to_group_median"] < 0.20)
    )
].copy()

print(f"Potential contextual price outliers (groups with >=5 listings): {len(price_flags):,}")
print("High contextual flags:", (price_flags["price_to_group_median"] > 5).sum())
print("Low contextual flags:", (price_flags["price_to_group_median"] < 0.20).sum())

print()
print("Most extreme high flags:")
display(price_flags.sort_values("price_to_group_median", ascending=False).head(10))

print()
print("Most extreme low flags:")
display(price_flags.sort_values("price_to_group_median").head(5))


Potential contextual price outliers (groups with >=5 listings): 31
High contextual flags: 23
Low contextual flags: 8

Most extreme high flags:


,make_name,model_name,trim_name,year,mileage,price,group_median_price,group_size,price_to_group_median
206696,Ford,Fusion,SE,2014,80306.0,999995.0,10495.0,112,95.282992
226088,Ford,Mustang,NaN,2015,385.0,299900.0,21985.0,128,13.641119
49102,Audi,A8,L 4.0T quattro Sport AWD,2017,33802.0,500000.0,43699.5,10,11.441779
145548,Honda,Pilot,EX 4WD,2011,164000.0,123456.0,10800.0,85,11.431111
193633,Ford,Mustang,NaN,2015,NaN,249900.0,21985.0,128,11.366841
252412,GMC,Acadia,NaN,2021,5.0,425460.0,44222.5,74,9.620894
239515,GMC,Acadia,NaN,2021,5.0,424060.0,44222.5,74,9.589236
300388,GMC,Acadia,NaN,2021,5.0,421780.0,44222.5,74,9.537679
33496,GMC,Acadia,NaN,2021,5.0,421780.0,44222.5,74,9.537679
143046,GMC,Acadia,NaN,2021,5.0,410160.0,44222.5,74,9.274917



Most extreme low flags:


,make_name,model_name,trim_name,year,mileage,price,group_median_price,group_size,price_to_group_median
155481,Nissan,Altima,2.5 S FWD,2020,NaN,386.9,23317.0,1671,0.016593
149849,Chevrolet,Impala,LT Fleet FWD,2012,NaN,999.0,6996.5,38,0.142786
229879,Chevrolet,Avalanche,1500 4WD,2003,121292.0,999.0,6900.0,9,0.144783
173285,Buick,LaCrosse,CXL FWD,2008,106589.0,999.0,6025.0,6,0.165809
273607,Chevrolet,Tahoe,LS RWD,2003,145035.0,999.0,5997.0,10,0.166583


Our review found that most vehicles over $200,000 were plausible for their make/model.  We retained them. Nine records had particularly strong internal evidence of price error: one 2020 Nissan Altima, one 2014 Ford Fusion, one 2017 Audi A8, and six 2021 GMC Acadias. We retain other statistically extreme prices when the available data do not clearly indicate an error.


In [19]:
price_error_mask = (
    # 2020 Nissan Altima: implausibly low compared with same-year Altimas
    (
        (df["make_name"] == "Nissan")
        & (df["model_name"] == "Altima")
        & (df["year"] == 2020)
        & (df["price"] < 1_000)
    )
    |
    # 2014 Ford Fusion: implausibly high relative to thousands of comparable Fusions
    (
        (df["make_name"] == "Ford")
        & (df["model_name"] == "Fusion")
        & (df["year"] == 2014)
        & (df["price"] > 100_000)
    )
    |
    # 2017 Audi A8: implausibly high relative to comparable A8 listings
    (
        (df["make_name"] == "Audi")
        & (df["model_name"] == "A8")
        & (df["year"] == 2017)
        & (df["price"] > 200_000)
    )
    |
    # 2021 GMC Acadia: six systematic high-price anomalies
    (
        (df["make_name"] == "GMC")
        & (df["model_name"] == "Acadia")
        & (df["year"] == 2021)
        & (df["price"] > 100_000)
    )
)

n_price_dropped = int(price_error_mask.sum())
df = df.loc[~price_error_mask].copy()

print(f"Dropped {n_price_dropped} contextually implausible price records")
print(f"Rows remaining: {len(df):,}")


Dropped 9 contextually implausible price records
Rows remaining: 296,585


### 3.3 Year
The observed range is checked for impossible model years. Older vehicles are statistical outliers but may be legitimate classic-car listings, so plausible years are retained. `vehicle_age` is engineered later as a more interpretable representation of the same information.


In [20]:
print(
    "Year range:", int(df["year"].min()), "-", int(df["year"].max()),
    "| Rows before 1990:", int((df["year"] < 1990).sum())
)


Year range: 1923 - 2021 | Rows before 1990: 444


### 3.4 Fuel-economy consistency
Notebook 1 showed that city values above 60 are overwhelmingly associated with electric vehicles, while a small number are labeled `Gasoline`. The repeated value 127 is investigated rather than automatically treated as a sentinel because MPGe values can legitimately repeat among electric vehicles. Only gasoline-labeled rows with city fuel economy above 60 are treated as inconsistent; their paired city/highway economy values are set to missing and later imputed from **training-only** fuel-type medians.


In [21]:
high_city_mpg = df.loc[
    df["city_fuel_economy"] > 60,
    ["make_name", "model_name", "year", "fuel_type", "city_fuel_economy", "highway_fuel_economy"]
].copy()

print("Fuel types among city_fuel_economy > 60:")
display(high_city_mpg["fuel_type"].value_counts(dropna=False))

mpg_127 = df.loc[df["city_fuel_economy"] == 127]
print(f"Rows with city_fuel_economy exactly 127: {len(mpg_127):,}")
print("Fuel types among exact-127 rows:")
display(mpg_127["fuel_type"].value_counts(dropna=False))

bad_mpg_mask = (
    (df["fuel_type"] == "Gasoline")
    & (df["city_fuel_economy"] > 60)
)
n_bad_mpg = int(bad_mpg_mask.sum())

df.loc[
    bad_mpg_mask,
    ["city_fuel_economy", "highway_fuel_economy"]
] = np.nan

print(f"Set paired fuel-economy values to NaN on {n_bad_mpg} inconsistent gasoline rows")


Fuel types among city_fuel_economy > 60:


fuel_type
Electric    1177
Gasoline      72
Name: count, dtype: int64

Rows with city_fuel_economy exactly 127: 436
Fuel types among exact-127 rows:


fuel_type
Electric    436
Name: count, dtype: int64

Set paired fuel-economy values to NaN on 72 inconsistent gasoline rows


### 3.5 `is_new` / mileage inconsistency
Notebook 1 found thousands of rows labeled `is_new=True` with mileage above 1,000. This is a real inconsistency, but mileage alone is not sufficient evidence to overwrite the source `is_new` field. The original flag is therefore retained. A separate `is_new_mileage_conflict` feature is engineered in Section 5 so the model can use the disagreement without silently rewriting the source data.


In [22]:
is_new_conflict_mask = df["is_new"].fillna(False) & (df["mileage"] > 1_000)
print(f"is_new=True with mileage > 1,000: {int(is_new_conflict_mask.sum()):,}")

if is_new_conflict_mask.any():
    print("Mileage distribution for conflicting rows:")
    display(df.loc[is_new_conflict_mask, "mileage"].describe())


is_new=True with mileage > 1,000: 4,404
Mileage distribution for conflicting rows:


count      4404.000000
mean       4361.449818
std        7355.726541
min        1001.000000
25%        2312.000000
50%        3391.500000
75%        4960.750000
max      351123.000000
Name: mileage, dtype: float64

### 3.6 Target completeness and 50% row-loss ceiling
Rows with a missing regression target cannot be used for supervised training, so any missing `price` rows are removed and counted toward the combined row-loss total. The assignment requires null + outlier row dropping to remain below 50% of the raw data.


In [23]:
missing_price_mask = df["price"].isna()
n_missing_price_dropped = int(missing_price_mask.sum())
if n_missing_price_dropped:
    df = df.loc[~missing_price_mask].copy()

n_after_row_drops = len(df)
total_dropped = n_raw - n_after_row_drops
pct_dropped = total_dropped / n_raw * 100

print(f"Dropped missing-target rows: {n_missing_price_dropped:,}")
print(f"Rows dropped so far: {total_dropped:,} of {n_raw:,} raw rows ({pct_dropped:.2f}%)")
assert pct_dropped < 50, "Combined null/outlier row dropping exceeds the 50% ceiling"


Dropped missing-target rows: 0
Rows dropped so far: 6,005 of 302,590 raw rows (1.98%)


### 3.7 Create the 80/20 split before learned preprocessing
We used an 80/20 split with `random_state=42`. We create that split **here**, before any imputation medians/modes, category-frequency selection, or scaling are learned. From this point forward, every learned preprocessing parameter is fit on `train_df` only and applied unchanged to `test_df`. This preserves the exact required split while avoiding train/test contamination.


In [24]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)
train_df = train_df.copy()
test_df = test_df.copy()

print(f"Preprocessing train split: {train_df.shape}")
print(f"Preprocessing test split:  {test_df.shape}")

# Prevent accidental reuse of the full cleaned dataframe in learned preprocessing.
cleaned_shape = df.shape
del df


Preprocessing train split: (237268, 50)
Preprocessing test split:  (59317, 50)


## 4. Imputation
Imputation is based on *why* data are missing. Constant/domain-defined fills are applied identically to both splits. Any median or mode that must be learned from data is calculated on the training split only and then applied to the test split.


### 4.1 History-report block
Missing history-report fields do not mean the vehicle was clean; they mean the report information was unavailable. `history_report_available` preserves that distinction before the six flags are filled with `False` for modeling.


In [25]:
history_block_present = [c for c in history_block_cols if c in train_df.columns]

for frame in [train_df, test_df]:
    frame["history_report_available"] = frame[history_block_present].notna().all(axis=1)
    for c in history_block_present:
        frame[c] = frame[c].fillna(False)

print("Added history_report_available; filled history-report flags with False where missing")


Added history_report_available; filled history-report flags with False where missing


### 4.2 Truck-only fields
`bed`/`bed_length` are structurally missing for many non-trucks. A `has_bed` indicator is captured before filling. Categorical truck fields use `Not Applicable`; numeric `bed_length` uses 0. Because `bed_length` was parsed in Section 2, it remains numeric rather than being one-hot encoded as a string measurement.


In [26]:
for frame in [train_df, test_df]:
    bed_signal = pd.Series(False, index=frame.index)
    if "bed" in frame.columns:
        bed_signal = bed_signal | frame["bed"].notna()
    if "bed_length" in frame.columns:
        bed_signal = bed_signal | frame["bed_length"].notna()
    frame["has_bed"] = bed_signal

    for c in ["bed", "cabin"]:
        if c in frame.columns:
            frame[c] = frame[c].fillna("Not Applicable")

    if "bed_length" in frame.columns:
        frame["bed_length"] = frame["bed_length"].fillna(0)

print("Added has_bed; filled truck-only fields")


Added has_bed; filled truck-only fields


### 4.3 Boolean mode fill
Any residual missing values in `is_new` or `franchise_dealer` are filled with the **training** mode. The same training-derived value is used on the test split.


In [27]:
for c in clean_bool_cols:
    if c in train_df.columns:
        mode_values = train_df[c].mode(dropna=True)
        fill_value = bool(mode_values.iloc[0]) if not mode_values.empty else False
        train_df[c] = train_df[c].fillna(fill_value)
        test_df[c] = test_df[c].fillna(fill_value)
        print(f"{c}: filled missing values with training mode = {fill_value}")


is_new: filled missing values with training mode = False
franchise_dealer: filled missing values with training mode = True


### 4.4 Training-only group-median helper
This helper learns group medians from `train_df`, applies them to both splits, and falls back to the training global median when a group cannot supply a value.


In [28]:
def _group_median_values(frame, medians, group_cols):
    if len(group_cols) == 1:
        return frame[group_cols[0]].map(medians)

    keys = pd.MultiIndex.from_frame(frame[group_cols])
    return pd.Series(medians.reindex(keys).to_numpy(), index=frame.index)


def fill_group_median_from_train(train, test, col, group_cols):
    group_cols = list(group_cols)
    medians = train.groupby(group_cols, dropna=False)[col].median()
    global_median = train[col].median()

    train_fill = _group_median_values(train, medians, group_cols)
    test_fill = _group_median_values(test, medians, group_cols)

    train[col] = train[col].fillna(train_fill).fillna(global_median)
    test[col] = test[col].fillna(test_fill).fillna(global_median)

    return global_median


### 4.5 Vehicle-aware numeric imputation
- Physical measurements are imputed by `body_type`.
- `mileage` and `owner_count` are imputed by `year` because both are age-related.
- `horsepower` is imputed using make/model/engine type, with a training global fallback.
- `engine_displacement` is imputed by `engine_type`.
- Fuel economy is imputed by `fuel_type`.

We used grouped strategies for domain-aware imputation rather than a blanket global median imputation.


In [29]:
# Physical measurements: bed_length is already handled structurally above.
physical_cols = [c for c in unit_cols if c != "bed_length" and c in train_df.columns]
for c in physical_cols:
    fill_group_median_from_train(train_df, test_df, c, ["body_type"])
print("Body-type median imputation:", physical_cols)

if "mileage" in train_df.columns:
    fill_group_median_from_train(train_df, test_df, "mileage", ["year"])

if "owner_count" in train_df.columns:
    fill_group_median_from_train(train_df, test_df, "owner_count", ["year"])

if "horsepower" in train_df.columns:
    fill_group_median_from_train(
        train_df, test_df, "horsepower", ["make_name", "model_name", "engine_type"]
    )

if "engine_displacement" in train_df.columns:
    fill_group_median_from_train(train_df, test_df, "engine_displacement", ["engine_type"])

for c in ["city_fuel_economy", "highway_fuel_economy"]:
    if c in train_df.columns:
        fill_group_median_from_train(train_df, test_df, c, ["fuel_type"])

print("Completed vehicle-aware numeric imputation")


Body-type median imputation: ['back_legroom', 'front_legroom', 'fuel_tank_volume', 'height', 'length', 'wheelbase', 'width', 'maximum_seating']
Completed vehicle-aware numeric imputation


### 4.6 Remaining numeric and categorical missingness
Any remaining numeric gaps (excluding `price`) use the training median. Remaining object-dtype gaps receive the explicit category `Unknown` rather than guessing a specific category. The test split receives the training numeric medians and the same constant categorical fill.


In [30]:
remaining_numeric = train_df.select_dtypes(include=[np.number]).columns.tolist()
remaining_numeric = [c for c in remaining_numeric if c != "price"]
filled_numeric = []

for c in remaining_numeric:
    if train_df[c].isna().any() or test_df[c].isna().any():
        fill_value = train_df[c].median()
        train_df[c] = train_df[c].fillna(fill_value)
        test_df[c] = test_df[c].fillna(fill_value)
        filled_numeric.append(c)

remaining_object = train_df.select_dtypes(include="object").columns.tolist()
filled_object = []
for c in remaining_object:
    if train_df[c].isna().any() or test_df[c].isna().any():
        train_df[c] = train_df[c].fillna("Unknown")
        test_df[c] = test_df[c].fillna("Unknown")
        filled_object.append(c)

print("Training-median filled remaining numeric columns:", filled_numeric)
print("Unknown-filled remaining object columns:", filled_object)


Training-median filled remaining numeric columns: ['seller_rating']
Unknown-filled remaining object columns: ['body_type', 'engine_type', 'exterior_color', 'franchise_make', 'fuel_type', 'interior_color', 'major_options', 'transmission', 'transmission_display', 'trim_name', 'wheel_system']


## 5. Feature Engineering
No engineered feature uses `price` or any direct function of `price`.


### 5.1 Vehicle age, mileage intensity, and `is_new` conflict
The dataset's validated maximum model year is 2021. `vehicle_age` is easier to interpret than raw calendar year, and keeping both would create perfect linear redundancy, so raw `year` is dropped after age-based features are created. `is_new_mileage_conflict` preserves the source inconsistency without overwriting `is_new`.


In [31]:
REFERENCE_YEAR = 2021

for frame in [train_df, test_df]:
    frame["vehicle_age"] = REFERENCE_YEAR - frame["year"]
    frame["mileage_per_year"] = frame["mileage"] / frame["vehicle_age"].clip(lower=1)
    frame["is_new_mileage_conflict"] = (
        frame["is_new"].astype(bool) & (frame["mileage"] > 1_000)
    )
    frame.drop(columns=["year"], inplace=True)

print("Added vehicle_age, mileage_per_year, and is_new_mileage_conflict; dropped raw year")


Added vehicle_age, mileage_per_year, and is_new_mileage_conflict; dropped raw year


### 5.2 Listing date
`listed_date` is parsed into month and day-of-week features. Any parse failures are filled using the **training** mode for each derived field, and the raw date string is then removed.


In [32]:
if "listed_date" in train_df.columns:
    for frame in [train_df, test_df]:
        parsed = pd.to_datetime(frame["listed_date"], errors="coerce")
        frame["listing_month"] = parsed.dt.month
        frame["listing_dow"] = parsed.dt.dayofweek

    for c in ["listing_month", "listing_dow"]:
        mode_values = train_df[c].mode(dropna=True)
        fill_value = mode_values.iloc[0] if not mode_values.empty else 0
        train_df[c] = train_df[c].fillna(fill_value)
        test_df[c] = test_df[c].fillna(fill_value)

    train_df.drop(columns=["listed_date"], inplace=True)
    test_df.drop(columns=["listed_date"], inplace=True)
    print("Parsed listed_date into listing_month and listing_dow")


Parsed listed_date into listing_month and listing_dow


### 5.3 `major_options`
`major_options` is a multi-label field rather than a single ordinary category. It is parsed into `n_options` and binary indicators for the ten options most common in the **training split**. The original `major_options` text is retained as metadata for the fairness/audit handoff and is excluded from modeling.


In [33]:
def parse_options(series):
    cleaned = (
        series.fillna("")
        .replace("Unknown", "")
        .str.strip("[]")
        .str.replace("'", "", regex=False)
        .str.split(",")
    )
    return cleaned.apply(lambda values: [v.strip() for v in values if v.strip()])

if "major_options" in train_df.columns:
    train_options = parse_options(train_df["major_options"])
    test_options = parse_options(test_df["major_options"])

    train_df["n_options"] = train_options.apply(len)
    test_df["n_options"] = test_options.apply(len)

    all_train_options = pd.Series(
        [opt for options in train_options for opt in options],
        dtype="object"
    )
    top_options = all_train_options.value_counts().head(10).index.tolist()

    for opt in top_options:
        col_name = "opt_" + re.sub(r"[^0-9a-zA-Z]+", "_", opt).strip("_").lower()
        train_df[col_name] = train_options.apply(lambda values, opt=opt: opt in values)
        test_df[col_name] = test_options.apply(lambda values, opt=opt: opt in values)

    print("Top training-set options encoded as flags:", top_options)


Top training-set options encoded as flags: ['Backup Camera', 'Bluetooth', 'Alloy Wheels', 'Heated Seats', 'Navigation System', 'Sunroof/Moonroof', 'CarPlay', 'Remote Start', 'Leather Seats', 'Blind Spot Monitoring']


## 6. Encoding
### 6.1 Label-encode Boolean columns
Every Boolean feature is mapped to 0/1 in both splits.


In [34]:
bool_cols = train_df.select_dtypes(include=["bool", "boolean"]).columns.tolist()

for c in bool_cols:
    train_df[c] = train_df[c].astype(int)
    test_df[c] = test_df[c].astype(int)

print(f"Label-encoded {len(bool_cols)} binary columns:")
print(bool_cols)


Label-encoded 23 binary columns:
['fleet', 'frame_damaged', 'franchise_dealer', 'has_accidents', 'isCab', 'is_cpo', 'is_new', 'is_oemcpo', 'salvage', 'theft_title', 'history_report_available', 'has_bed', 'is_new_mileage_conflict', 'opt_backup_camera', 'opt_bluetooth', 'opt_alloy_wheels', 'opt_heated_seats', 'opt_navigation_system', 'opt_sunroof_moonroof', 'opt_carplay', 'opt_remote_start', 'opt_leather_seats', 'opt_blind_spot_monitoring']


### 6.2 One-hot encode ordinary categorical columns using training frequencies
For each ordinary object-dtype categorical feature, the most frequent **training** category is the reference. Dummies are created for only the next 20 most frequent training categories, exactly as required by the prompt. Rare and unseen test categories receive all-zero dummies for that feature. The original object columns remain untouched as metadata.

`major_options` is excluded from this generic one-hot step because it is a multi-label field already parsed in Section 5.3; its raw object version is retained as metadata.


In [35]:
metadata_cols = train_df.select_dtypes(include="object").columns.tolist()
categorical_cols = [c for c in metadata_cols if c != "major_options"]

encoding_log = []

for col in categorical_cols:
    counts = train_df[col].value_counts(dropna=False)
    reference = counts.index[0]
    kept = counts.index[1:21].tolist()

    train_base = train_df[col].where(train_df[col].isin(kept))
    test_base = test_df[col].where(test_df[col].isin(kept))

    train_dummies = pd.get_dummies(train_base, prefix=col, dtype=int)
    test_dummies = pd.get_dummies(test_base, prefix=col, dtype=int)
    test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

    train_df = pd.concat([train_df, train_dummies], axis=1)
    test_df = pd.concat([test_df, test_dummies], axis=1)

    encoding_log.append({
        "column": col,
        "reference_category": reference,
        "n_dummies": train_dummies.shape[1],
    })

encoding_log_df = pd.DataFrame(encoding_log)
display(encoding_log_df)


,column,reference_category,n_dummies
0,city,Houston,20
1,bed,Not Applicable,3
2,body_type,SUV / Crossover,9
3,cabin,Not Applicable,4
4,engine_type,I4,20
5,exterior_color,Black,20
6,franchise_make,Unknown,20
7,fuel_type,Gasoline,7
8,interior_color,Black,20
9,listing_color,WHITE,14


## 7. Preserving Subgroup Metadata
All original object-dtype columns are retained in both exports for Notebook 5. These columns are metadata only and must be excluded when constructing `X` in Notebooks 3 and 4.


In [36]:
# Recompute after feature engineering; includes major_options metadata.
metadata_cols = train_df.select_dtypes(include="object").columns.tolist()

print(f"{len(metadata_cols)} metadata columns retained (exclude from modeling X/y):")
print(metadata_cols)


18 metadata columns retained (exclude from modeling X/y):
['city', 'bed', 'body_type', 'cabin', 'engine_type', 'exterior_color', 'franchise_make', 'fuel_type', 'interior_color', 'listing_color', 'major_options', 'make_name', 'model_name', 'sp_name', 'transmission', 'transmission_display', 'trim_name', 'wheel_system']


## 8. Scaling
The prompt requires standard scaling of all non-metadata features except `price`. The scaler is fit on **training features only** and then applied unchanged to the test split. This includes binary/dummy features because the prompt's wording says all non-metadata features.


In [37]:
assert train_df.columns.tolist() == test_df.columns.tolist(), "Train/test columns differ before scaling"

feature_cols = [
    c for c in train_df.columns
    if c not in metadata_cols and c != "price"
]

non_numeric = train_df[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()
assert not non_numeric, f"Non-numeric modeling features remain: {non_numeric}"

train_feature_nans = train_df[feature_cols].columns[train_df[feature_cols].isna().any()].tolist()
test_feature_nans = test_df[feature_cols].columns[test_df[feature_cols].isna().any()].tolist()
assert not train_feature_nans, f"NaNs remain in training features: {train_feature_nans}"
assert not test_feature_nans, f"NaNs remain in test features: {test_feature_nans}"

scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

print(f"Standard-scaled {len(feature_cols)} feature columns")
print(f"Left untouched: price and {len(metadata_cols)} metadata columns")


Standard-scaled 293 feature columns
Left untouched: price and 18 metadata columns


## 9. Split and Export
The 80/20 split was created in Section 3.7 before learned preprocessing. No re-splitting occurs here; the exact processed train/test rows are exported for Notebook 3.


In [38]:
train_path = OUTPUT_DIR / "processed_train.csv"
test_path = OUTPUT_DIR / "processed_test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Train: {train_df.shape} -> {train_path}")
print(f"Test:  {test_df.shape} -> {test_path}")
print(f"Train fraction: {len(train_df) / (len(train_df) + len(test_df)):.3f}")


Train: (237268, 312) -> C:\Users\india\OneDrive\Desktop\MSDS 565\MSDS565-CarSales-DataStars\data\processed_train.csv
Test:  (59317, 312) -> C:\Users\india\OneDrive\Desktop\MSDS 565\MSDS565-CarSales-DataStars\data\processed_test.csv
Train fraction: 0.800


## 10. Sanity Checks Before Notebook 3
These checks verify the requirements most likely to break downstream modeling or the Notebook 5 fairness audit.


In [39]:
# No missing values anywhere in the exported splits.
train_nan_cols = train_df.columns[train_df.isna().any()].tolist()
test_nan_cols = test_df.columns[test_df.isna().any()].tolist()
assert not train_nan_cols, f"NaNs remain in train: {train_nan_cols}"
assert not test_nan_cols, f"NaNs remain in test: {test_nan_cols}"
print("No missing values in either processed split")

# price remains in original dollar units.
assert train_df["price"].median() > 1_000, "price looks scaled or otherwise altered"
assert test_df["price"].median() > 1_000, "price looks scaled or otherwise altered"
print("Price remains in original dollar units")

# Metadata is present in both exports.
missing_meta_train = [c for c in metadata_cols if c not in train_df.columns]
missing_meta_test = [c for c in metadata_cols if c not in test_df.columns]
assert not missing_meta_train and not missing_meta_test, "Metadata columns missing from an export"
print(f"All {len(metadata_cols)} metadata columns are present in both splits")

# Leakage / ID / temporary target-derived diagnostics are absent.
forbidden_cols = [
    "savings_amount", "listing_id", "sp_id", "Unnamed: 0",
    "price_to_group_median", "group_median_price"
]
forbidden_present = [c for c in forbidden_cols if c in train_df.columns or c in test_df.columns]
assert not forbidden_present, f"Forbidden/leakage columns present: {forbidden_present}"
print("No ID, leakage, or temporary target-derived diagnostic columns in exports")

# Same columns/order in both splits.
assert train_df.columns.tolist() == test_df.columns.tolist(), "Train/test export columns differ"

# All modeling features are numeric; object columns are metadata only.
model_non_numeric = train_df[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()
assert not model_non_numeric, f"Non-numeric modeling columns remain: {model_non_numeric}"
print("All modeling features are numeric")


No missing values in either processed split
Price remains in original dollar units
All 18 metadata columns are present in both splits
No ID, leakage, or temporary target-derived diagnostic columns in exports
All modeling features are numeric


### Optional relationship check
Standard scaling is linear, so the direction and approximate strength of the `horsepower`/`price` relationship should remain recognizable after preprocessing. Imputation can move the exact value slightly.


In [40]:
if "horsepower" in train_df.columns:
    r = train_df["horsepower"].corr(train_df["price"])
    print(f"horsepower vs price correlation after preprocessing: {r:.3f}")


horsepower vs price correlation after preprocessing: 0.582


## 11. Summary for Notebook 3
- **Target:** `price`, retained in original dollar units.
- **Exact split:** 80/20 with `random_state=42`, created before learned preprocessing and exported unchanged.
- **Metadata:** all object-dtype metadata columns remain in both CSVs and must be excluded from modeling `X`.
- **Model features:** all remaining non-metadata columns are numeric and training-fitted standard scaling has already been applied.
- **No target leakage:** `savings_amount` and temporary price-diagnostic fields are absent; no engineered predictor uses `price`.
- **Notebook 3 rule:** read `processed_train.csv` and `processed_test.csv`, set `y = price`, and drop `price` plus all object-dtype metadata columns from `X`.
